# Hyperparameter transfer

## Basic imports

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import math
import time
import numpy as np

from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
import os
!env

HOMEBREW_PREFIX=/opt/homebrew
PYTHONUNBUFFERED=1
HOMEBREW_NO_INSTALL_CLEANUP=1
VIRTUALENVWRAPPER_WORKON_CD=1
PYDEVD_IPYTHON_COMPATIBLE_DEBUGGING=1
PYTHONIOENCODING=UTF-8
LDFLAGS=-L/opt/homebrew/opt/ffmpeg-full/lib
HOMEBREW_REPOSITORY=/opt/homebrew
OLLAMA_API_KEY=ae3fc51629cd49f7a69b192854482d1c.WfekVByo8nSSrKdpwm1EKh2S
VIRTUALENVWRAPPER_PROJECT_FILENAME=.project
PATH=/Users/deven/.virtualenvs/Machine_Learning_Algorithms/bin:/opt/anaconda3/bin:/Library/Frameworks/Python.framework/Versions/3.13/bin:/Library/Frameworks/Python.framework/Versions/3.12/bin:/opt/homebrew/bin:/opt/homebrew/sbin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/pkg/env/global/bin:/Applications/quarto/bin:/opt/homebrew/opt/ffmpeg-full/bin:/opt/anaconda3/bin:/opt/home

## Question 1: Spectral norm of Gaussian
 (Spectral norm of a Gaussian matrix; 1pt) Sample a d×dmatrix with entries drawn
iid NORMAL(0,1). Vary  d and see how the spectral norm varies. Can you guess a
scaling rule for the spectral norm of iid NORMAL(0,1) matrices?—i.e. find or fit two
coefficients α,β >0 such that the spectral norm is well-approximated by α·dβ
.


In [58]:
d = 5000

M = torch.randn(size=(d+10,d+10),device="mps")
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

141.3922119140625


## Question 2: Spectral norm of orthogonal matrix
7. (Spectral norm of an orthogonal matrix; 1pt) In PyTorch, sample a d×drandom
orthogonal matrix and compute its spectral norm. Do this for varying d. What do
you notice?
Ansers
1.   List item

1.   List item
2.   List item


2.   List item



In [46]:
from torch.nn.init import orthogonal_

d = 2000

M = torch.zeros(size=(d,d),device="mps")
orthogonal_(M) # this line resamples M to be a random semi-orthogonal matrix
spec_norm = torch.linalg.matrix_norm(M, ord=2)

print(spec_norm.item())

1.0000003576278687


## Question 3: Power iteration

In [60]:
def spectral_norm(A, n_steps=1000):
    v = torch.randn(A.shape[1], device=A.device)
    for _ in range(n_steps):
        v /= v.norm()
        v = A @ v @ A
    return v.norm().sqrt()

d = 2000

M = torch.randn(size=(d,d), device="mps")

torch.mps.synchronize()
t0 = time.time()
spec_norm = spectral_norm(M).item()

t = time.time()-t0
print(f"computed {spec_norm=} in {t=} seconds")

torch.mps.synchronize()
t0 = time.time()
spec_norm = torch.linalg.matrix_norm(M,ord="fro").item()
t = time.time()-t0
print(f"computed {spec_norm=} in {t=} seconds")

computed spec_norm=199.7342071533203 in t=7.335444927215576 seconds


In [68]:
#torch.rand((1,2,3,4,5,6,7,8,9,10),device="mps")

KeyboardInterrupt: 

## Question 4: Learning rate transfer across width and depth
You only need to modify the two blocks of code that are marked, and then choose the learning rate `eta` for the large width training run.

In [83]:
batch_size = 128

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean, std)])

trainset = datasets.CIFAR100('./data', train=True,  download=True, transform=transform)
testset  = datasets.CIFAR100('./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True,  pin_memory=True)
test_loader  = torch.utils.data.DataLoader(testset,  batch_size=batch_size, shuffle=False, pin_memory=True)

## Define the MLP architecture

In [84]:
class MLP(nn.Module):
    def __init__(self, depth, width):
        super(MLP, self).__init__()

        self.initial = nn.Linear(3072, width, bias=False)
        self.layers = nn.ModuleList([nn.Linear(width, width, bias=False) for _ in range(depth-2)])
        self.final = nn.Linear(width, 10, bias=False)

        self.nonlinearity = lambda x: F.relu(x) * math.sqrt(2)

    def forward(self, x):
        x = x.view(x.shape[0],-1)

        x = self.initial(x)
        x = self.nonlinearity(x)

        for layer in self.layers:
            x = layer(x)
            x = self.nonlinearity(x)

        return self.final(x)

## Define the train and test loop

In [85]:
mlp = MLP(5,4096)
print(mlp.to("mps"))

MLP(
  (initial): Linear(in_features=3072, out_features=4096, bias=False)
  (layers): ModuleList(
    (0-2): 3 x Linear(in_features=4096, out_features=4096, bias=False)
  )
  (final): Linear(in_features=4096, out_features=10, bias=False)
)


In [88]:
def loop(net, train, eta):
    dataloader  = train_loader   if train else test_loader
    description = "Training... " if train else "Testing... "

    acc_list = []

    for data, target in tqdm(dataloader, desc=description):
        data, target = data.to("mps"), target.to("mps")
        output = net(data)
        #print(data)

        loss = output.logsumexp(dim=1).mean() - output[range(target.shape[0]),target].mean() # cross-entropy loss
        acc = (output.max(dim=1)[1] == target).sum() / target.shape[0] # accuracy
        #print(acc.item())
        acc_list.append(acc.item())

        if train:
            loss.backward()

            for p in net.parameters():
                print(p.shape)
                fan_out, fan_in = p.shape
                update = torch.sign(p.grad)
                #print(update)
                ## START BLOCK that you should modify
                update *=(fan_out/fan_in)
                ## END BLOCK that you should modify
                p.data -= eta * update
            net.zero_grad()

    return np.mean(acc_list)

## Sweep the learning rate at small width



In [89]:
depth = 5
width = 32

import torch

def initialize_matrix(p):
    """
    Initializes a 2D weight matrix parameter following:
    W_k = (d_k / d_{k-1}) * M_k, where M_k is a random semi-orthogonal matrix.
    """
    if p.dim() < 2:
        # Safeguard: Fallback to standard uniform initialization for 1D biases
        torch.nn.init.uniform_(p.data, -0.1, 0.1)
        return

    # Extract dimensions: d_k (fan_out), d_{k-1} (fan_in)
    d_k, d_k1 = p.shape

    # 1. Sample a random semi-orthogonal matrix M_k directly on Apple Silicon GPU
    M_k = torch.empty(d_k, d_k1, device="mps")
    torch.nn.init.orthogonal_(M_k)

    # 2. Compute scale factor: (d_k / d_{k-1})
    scale_factor = d_k / d_k1

    # 3. Set the final weight matrix: W_k = scale_factor * M_k
    p.data.copy_(scale_factor * M_k)

for eta in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    print(f"Training at {width=}, {depth=}, {eta=}")

    net = MLP(depth, width).to(device="mps")

    print("\nNetwork tensor shapes are:\n")
    for name, p in net.named_parameters():
        print(p.shape, '\t', name)
        initialize_matrix(p)

    train_acc = loop(net, train=True,  eta=eta)
    test_acc  = loop(net, train=False, eta=None)

    print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
    print("===================================================================\n")

Training at width=32, depth=5, eta=0.0001

Network tensor shapes are:

torch.Size([32, 3072]) 	 initial.weight
torch.Size([32, 32]) 	 layers.0.weight
torch.Size([32, 32]) 	 layers.1.weight
torch.Size([32, 32]) 	 layers.2.weight
torch.Size([10, 32]) 	 final.weight


Training... :   0%|          | 1/391 [00:00<01:01,  6.32it/s]

torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])


Training... :   4%|▍         | 16/391 [00:00<00:05, 72.72it/s]

torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32

...

torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32

Training... : 100%|██████████| 391/391 [00:02<00:00, 150.48it/s]


torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([32, 32])
torch.Size([10, 32])
torch.Size([32, 3072])
torch.Size([32

Testing... : 100%|██████████| 79/79 [00:00<00:00, 178.29it/s]


We achieved train acc=0.100 and test acc=0.100




## Run the best learning rate at large width

In [91]:
depth = 5
width = 4096

# START BLOCK you should set this to the best value of eta from the previous cell
eta = 0.001
# END BLOCK

print(f"Training at {width=}, {depth=}, {eta=}")

net = MLP(depth, width).to("mps")

print("\nNetwork tensor shapes are:\n")
for name, p in net.named_parameters():
    print(p.shape, '\t', name)
    initialize_matrix(p)

train_acc = loop(net, train=True,  eta=eta)
test_acc  = loop(net, train=False, eta=None)

print(f"\nWe achieved train acc={train_acc:.3f} and test acc={test_acc:.3f}\n")
print("===================================================================\n")

Training at width=4096, depth=5, eta=0.001

Network tensor shapes are:

torch.Size([4096, 3072]) 	 initial.weight
torch.Size([4096, 4096]) 	 layers.0.weight
torch.Size([4096, 4096]) 	 layers.1.weight
torch.Size([4096, 4096]) 	 layers.2.weight
torch.Size([10, 4096]) 	 final.weight


Training... :   1%|          | 4/391 [00:00<00:21, 17.90it/s]

torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])


Training... :   3%|▎         | 12/391 [00:00<00:14, 26.91it/s]

torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])


...

torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])
torch.Size([4096

Training... : 100%|██████████| 391/391 [00:12<00:00, 30.31it/s]


torch.Size([4096, 3072])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([4096, 4096])
torch.Size([10, 4096])


Testing... : 100%|██████████| 79/79 [00:00<00:00, 96.86it/s] 


We achieved train acc=0.289 and test acc=0.291


